# 5.9 每次實驗用了多少資源？

# 第 5 章：訓練與評估形成一個流程

前置：第4章的 logits／shift／loss。程式預設只做結構、梯度與記錄檢查。凡是涉及學到能力的對照，先給可執行配方，之後由讀者在 GPU 上跑；不會把隨機模型的輸出寫成訓練成果。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：一次更新的生命週期**

忘記清零會累加；累積梯度時才刻意保留。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/backward.svg")))

**先想一想：**相同參數量會有相同執行時間嗎？

先量清楚參數與本機forward耗時，再比較模型。GPU操作是非同步的，GPU計時前後需同步；CPU不用。不能把一次微小耗時當穩定benchmark。

**把直覺寫成數學：**FP32參數payload約參數數量×4 bytes；不包含optimizer與activation。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
import time
from tiny_perceptron.model import TinyLM, ModelConfig

model = TinyLM(ModelConfig(width=8))
x = torch.tensor([[1, 2, 3]])
model(x)
start = time.perf_counter()
for _ in range(10):
    model(x)
print(model.description(), "CPU每次forward秒", (time.perf_counter() - start) / 10)

**如何讀結果：**固定硬體、dtype、序列長度與warmup；這裡只記實測，不寫跨硬體速度結論。

**只改一件事：**只把T從3改成6。
